# DataCleans AI — Customer Churn Analytical Case Study

**Author:** Junior Data Analyst & Python Developer Portfolio  
**Objective:** End-to-end demonstration of data profiling, automated data cleaning, statistical anomaly treatment, validation, and exploratory data analysis (EDA) using the `src/` modular library.

## 1. Setup & Environment

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in Python module search path
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

import pandas as pd
import numpy as np
from src.data_loader import DataLoader
from src.profiler import DataProfiler
from src.cleaner import DataCleaner
from src.validator import DataValidator
from src.visualizer import DataVisualizer
from src.report_generator import ReportGenerator
from src.database import DatabaseManager

## 2. Ingest the Raw Dirty Dataset
We load `customer_churn_dirty.csv`, which intentionally contains duplicates, whitespace inconsistencies, mixed date formats, currency strings, missing cells, and age outliers.

In [ ]:
raw_csv_path = ROOT_DIR / "data" / "samples" / "customer_churn_dirty.csv"
df_raw = DataLoader.load_dataset(raw_csv_path)
print(f"Raw Dataset Loaded: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns.")
df_raw

## 3. Data Quality Health Profiling
Inspect completeness, duplication, and baseline health score.

In [ ]:
baseline_summary = DataProfiler.get_basic_summary(df_raw)
baseline_health = DataProfiler.calculate_health_score(df_raw)
baseline_summary["health_score"] = baseline_health

print(f"Baseline Health Score: {baseline_health} / 100")
for k, v in baseline_summary.items():
    print(f"  - {k}: {v}")

In [ ]:
# Detailed per-column profile
col_profile = DataProfiler.get_column_profiles(df_raw)
col_profile

## 4. End-to-End Cleaning Pipeline
Execute modular transformations to resolve identified data flaws non-destructively.

In [ ]:
cleaner = DataCleaner(df_raw)

# Step 1: Remove duplicate rows
cleaner.remove_duplicates(subset=None, keep="first")

# Step 2: Trim whitespace across all string columns
cleaner.strip_whitespace()

# Step 3: Clean currency symbols & sentinel nulls in charges
cleaner.clean_currency_and_numbers(["monthly_charges", "total_charges"])

# Step 4: Standardize dates to YYYY-MM-DD
cleaner.parse_mixed_dates("join_date", target_format="%Y-%m-%d")

# Step 5: Standardize categorical booleans (churn)
cleaner.standardize_booleans("churn")

# Step 6: Map inconsistent gender labels ('M', 'MALE' -> 'Male'; 'F', 'female' -> 'Female')
cleaner.map_values("gender", {"M": "Male", "MALE": "Male", "male": "Male", "F": "Female", "female": "Female"})

# Step 7: Impute missing categorical values
cleaner.impute_missing("gender", strategy="mode")
cleaner.impute_missing("payment_method", strategy="mode")

# Step 8: Impute missing numerical charges & date
cleaner.impute_missing("monthly_charges", strategy="median")
cleaner.impute_missing("total_charges", strategy="median")
cleaner.impute_missing("join_date", strategy="bfill")

# Step 9: Treat age outliers (bounds 18 to 90)
cleaner.treat_outliers("age", method="iqr", action="clip")

df_cleaned = cleaner.df.copy()
print("Cleaning pipeline completed! Audit trail steps:")
for step in cleaner.get_audit_trail():
    print(f"  ✓ {step['operation']}: {step}")

## 5. Post-Cleaning Quality Validation

In [ ]:
# Verify zero nulls
val_nulls = DataValidator.validate_no_nulls(df_cleaned)
print(f"Zero Nulls Validation: {val_nulls['is_valid']}")

# Verify unique customer ID
val_unique = DataValidator.validate_unique(df_cleaned, "customer_id")
print(f"Customer ID Unique Validation: {val_unique['is_valid']}")

# Verify age range
val_age = DataValidator.validate_numeric_range(df_cleaned, "age", min_value=18, max_value=100)
print(f"Age Range Validation: {val_age['is_valid']}")

# Verify gender categories
val_gender = DataValidator.validate_allowed_categories(df_cleaned, "gender", {"Male", "Female"})
print(f"Gender Categories Validation: {val_gender['is_valid']}")

## 6. Pre vs. Post Cleaning Health Score Comparison

In [ ]:
cleaned_summary = DataProfiler.get_basic_summary(df_cleaned)
cleaned_health = DataProfiler.calculate_health_score(df_cleaned)
cleaned_summary["health_score"] = cleaned_health

print(f"Raw Health Score:     {baseline_health} / 100")
print(f"Cleaned Health Score: {cleaned_health} / 100 (Gain: +{round(cleaned_health - baseline_health, 1)})")
df_cleaned

## 7. Export Cleaned Dataset & Audit Trail

In [ ]:
out_path = ROOT_DIR / "data" / "processed" / "customer_churn_cleaned.csv"
DataLoader.save_dataset(df_cleaned, out_path)
print(f"Exported cleaned dataset to: {out_path}")

# Generate report
report_md = ReportGenerator.generate_markdown_report(
    dataset_name="customer_churn_dirty.csv",
    initial_metrics=baseline_summary,
    cleaned_metrics=cleaned_summary,
    audit_trail=cleaner.get_audit_trail(),
)
report_file = ROOT_DIR / "reports" / "customer_churn_audit_report.md"
report_file.write_text(report_md, encoding="utf-8")
print(f"Saved audit report to: {report_file}")